# 05 · Defect classification — ResNet50 vs EfficientNet-B0

Fine-tunes ImageNet-pretrained ResNet50 and EfficientNet-B0 on the ROI crops (224×224, grayscale→3 channels, `WeightedRandomSampler` for class balance), then reports accuracy, precision, recall, F1 and confusion matrices on the validation split.

| | |
|---|---|
| **Input** | `ImageFolder` dataset produced by notebook `04` |
| **Output** | `<model>_solar_final.pth` (not committed), metrics, confusion matrices |
| **Runs on** | GPU (Google Colab) |

> **Notes**
> - Reported numbers are the **last-epoch** validation metrics (15 epochs, Adam, lr 5e-4, StepLR).
> - Weights and datasets are not included in this repository (see `DATA.md`).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torch.optim import lr_scheduler
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Core settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = '/content/drive/MyDrive/inspectra/classification_dataset'
CLASS_NAMES = ['Black_Spot', 'Cell_Scratch', 'Cell_Difference', 'Micro_Crack']
BATCH_SIZE = 32
NUM_EPOCHS = 15

In [ ]:
# Preprocessing / augmentation pipeline
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3), # 3 channels, to match the pretrained models
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1), # improves robustness to contrast changes
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load the data
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'valid']}

# WeightedRandomSampler to balance the classes
targets = torch.tensor(image_datasets['train'].targets)
class_sample_count = np.array([len(np.where(targets == t)[0]) for t in range(len(CLASS_NAMES))])
weight = 1. / class_sample_count
samples_weight = torch.tensor([weight[t] for t in targets])

sampler = WeightedRandomSampler(weights=samples_weight, num_samples=len(samples_weight), replacement=True)

# DataLoaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, sampler=sampler),
    'valid': DataLoader(image_datasets['valid'], batch_size=BATCH_SIZE, shuffle=False)
}

In [ ]:
def initialize_model(model_name, num_classes=4):
    if model_name == 'resnet':
        model = models.resnet50(weights='DEFAULT')
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == 'efficientnet':
        model = models.efficientnet_b0(weights='DEFAULT')
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_ftrs, num_classes)
        )
    return model.to(DEVICE)

In [ ]:
def train_model_comprehensive(model, criterion, optimizer, scheduler, num_epochs=15):
    # Containers that record the metrics for analysis
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss, running_corrects = 0.0, 0

            pbar = tqdm(dataloaders[phase], desc=f"Epoch {epoch+1} {phase}")
            for inputs, labels in pbar:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train': scheduler.step()

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            # Store the results in the history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

            print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    return model, history

In [ ]:
all_results = {}
model_names = ['resnet', 'efficientnet']

for name in model_names:
    print(f"\n" + "="*30)
    print(f"🔥 Training {name.upper()}")
    print("="*30)

    # 1. Initialise the model
    current_model = initialize_model(name)

    # 2. Set up the optimiser and the LR scheduler
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(current_model.parameters(), lr=0.0005)
    exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # 3. Train
    trained_m, history = train_model_comprehensive(
        current_model, criterion, optimizer, exp_lr_scheduler, num_epochs=NUM_EPOCHS
    )

    # 4. Save the model and the results
    all_results[name] = {'model': trained_m, 'history': history}
    torch.save(trained_m.state_dict(), f'{name}_solar_final.pth')

In [ ]:
plt.figure(figsize=(15, 6))
for name in model_names:
    plt.plot(all_results[name]['history']['val_acc'], label=f'{name} Val Acc')
    plt.plot(all_results[name]['history']['train_acc'], '--', label=f'{name} Train Acc', alpha=0.5)

plt.title('Comparison: Training vs Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()




## Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

def show_results(model, dataloader, classes, model_name):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Confusion Matrix
    plt.figure(figsize=(10, 7))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Classification Report
    print(f"\n📝 Detailed Report for {model_name}:")
    print(classification_report(all_labels, all_preds, target_names=classes))

    # Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print(f"\n====== {model_name} Classification Metrics ======")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("============================================")

In [ ]:
for name in model_names:
    print(f"\n{'='*50}")
    print(f"Evaluating {name.upper()} Model")
    print(f"{'='*50}")
    show_results(all_results[name]['model'], dataloaders['valid'], CLASS_NAMES, name.upper())